In [55]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import warnings; warnings.simplefilter('ignore')
import sys
import h5py
import pandas as pd
import seaborn as sns
sys.path.insert(0, '/Users/jsmonzon/Research/SatGen/mcmc/src/')
import jsm_ancillary
import jsm_visualize
import jsm_SHMR
import jsm_mcmc
import jsm_stats
import jsm_models
import evolve as ev
import galhalo as gh
import profiles as profiles
import config as cfg
import os
import cosmo as co

In [57]:
plt.style.use('../../../SatGen/notebooks/paper1/paper.mplstyle')
double_textwidth = 7.0 #inches
single_textwidth = 3.5 #inches
levelz = [1-0.99, 1-0.95, 1-0.68]

In [63]:
trees = []


mass_bins = [13.0, 13.2, 13.4, 13.6, 13.8, 14.0]

for i, mvir in enumerate(mass_bins):

    filepath = f"../../data/local_trees/evolved_trees/tree_{mvir:.1f}.npz"

    m = jsm_visualize.Arborist(
        file=filepath,
        merger_crit=-2,
        verbose=False)
    m.dendrochronology(mass_threshold=7.75e10, kmax=2)
    m.canopy(verbose=True)
    
    trees.append(m)

-----------------
at z=0, withering: Nsub=[4 4 0], fsub=[0.21283336 0.        ], MMs=[1.80971611e+12 1.80971611e+12 0.00000000e+00]
at z=0, rvir: Nsub=[4 4 0], fsub=[0.21283336 0.        ], MMs=[1.80971611e+12 1.80971611e+12 0.00000000e+00]
at z=0, artificial: Nsub=[4 4 0], fsub=[0.21283336 0.        ], MMs=[1.80971611e+12 1.80971611e+12 0.00000000e+00]
-----------------
at z=0, withering: Nsub=[7 7 0], fsub=[0.11114521 0.        ], MMs=[5.89910296e+11 5.89910296e+11 0.00000000e+00]
at z=0, rvir: Nsub=[2 2 0], fsub=[0.01769693 0.        ], MMs=[1.75748814e+11 1.75748814e+11 0.00000000e+00]
at z=0, artificial: Nsub=[2 2 0], fsub=[0.01769693 0.        ], MMs=[1.75748814e+11 1.75748814e+11 0.00000000e+00]
-----------------
at z=0, withering: Nsub=[11 10  1], fsub=[0.28752984 0.01081828], MMs=[3.59505708e+12 3.59505708e+12 2.71742888e+11]
at z=0, rvir: Nsub=[6 5 1], fsub=[0.22635195 0.01081828], MMs=[3.59505708e+12 3.59505708e+12 2.71742888e+11]
at z=0, artificial: Nsub=[6 5 1], fsub=[0.22

In [90]:
m = jsm_visualize.Arborist(
    file="../../data/local_trees/evolved_trees/tree_14.0.npz",
    merger_crit=-2,
    verbose=False)
m.dendrochronology(mass_threshold=1e10, kmax=2)
m.canopy(verbose=True)

-----------------
at z=0, withering: Nsub=[265 257   8], fsub=[0.18222852 0.00342435], MMs=[1.00683327e+12 1.00683327e+12 1.63119003e+11]
at z=0, rvir: Nsub=[140 136   4], fsub=[0.10330019 0.00230124], MMs=[1.00683327e+12 1.00683327e+12 1.63119003e+11]
at z=0, artificial: Nsub=[124 120   4], fsub=[0.09451392 0.00230124], MMs=[1.00683327e+12 1.00683327e+12 1.63119003e+11]


In [62]:
# color mapping for each order k
K_COLORS = ["C0", "C1", "C2", "C3", "C4", "C5"]

def plot_host_panel_Nsub(ax, tree_instance):

    # --- total counts ---
    ax.step(tree_instance.CosmicTime, tree_instance.Nsub_withering[:, 0], where="mid", color="C0", lw=2, label="above m$_{\\rm thresh}$")
    ax.step(tree_instance.CosmicTime, tree_instance.Nsub_rvir[:, 0], where="mid", color="C1", lw=2, label="within R$_{\\rm vir}$")
    ax.step(tree_instance.CosmicTime, tree_instance.Nsub_artificial[:, 0], where="mid", color="C2", lw=2, label="doesn't disrupt")

    # --- per order k ---
    for k in range(1, 3):
        ax.step(
            tree_instance.CosmicTime,
            tree_instance.Nsub_withering[:, k],
            where="mid",
            color="C0",
            lw=0.5, label="k="+str(k), ls="--")

def plot_host_panel_fsub(ax, tree_instance):

    # --- total counts ---
    ax.step(tree_instance.CosmicTime, tree_instance.fsub_withering[:, 0], where="mid", color="C0", lw=2, label="above m$_{\\rm thresh}$")
    ax.step(tree_instance.CosmicTime, tree_instance.fsub_rvir[:, 0], where="mid", color="C1", lw=2, label="within R$_{\\rm vir}$")
    ax.step(tree_instance.CosmicTime, tree_instance.fsub_artificial[:, 0], where="mid", color="C2", lw=2, label="doesn't disrupt")

    # --- per order k ---
    for k in range(1, 2):
        ax.step(
            tree_instance.CosmicTime,
            tree_instance.fsub_withering[:, k],
            where="mid",
            color="C0",
            lw=0.5, label="k="+str(k+1), ls="--")



In [ ]:
fig, ax = plt.subplots(
    2, 3,
    figsize=(2 * double_textwidth, double_textwidth),
    sharex=True,
    sharey=True
)

mass_bins = [13.0, 13.2, 13.4, 13.6, 13.8, 14.0]

# --------------------------------------------------
# Fill panels
# --------------------------------------------------
for i, mvir in enumerate(mass_bins):

    row = i // 3
    col = i % 3

    plot_host_panel_Nsub(ax[row, col],trees[i])

    ax[row, col].text(0.02, 0.97, rf"$\log M_{{\rm vir}} = {mvir:.1f}$",
        transform=ax[row, col].transAxes,
        va='top', ha='left')
# --------------------------------------------------
# Axis labels
# --------------------------------------------------
for a in ax[-1, :]:
    a.set_xlabel('cosmic time [Gyr]')

for a in ax[:, 0]:
    a.set_ylabel('N$_{\\rm sub}$')

# --------------------------------------------------
# X limits (adjust as appropriate for your zsample)
# --------------------------------------------------
ax[0,0].set_xlim(0, 10)
ax[0,0].legend(loc=1)
# --------------------------------------------------
# Final layout
# --------------------------------------------------
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    2, 3,
    figsize=(2 * double_textwidth, double_textwidth),
    sharex=True,
    sharey=True
)

mass_bins = [13.0, 13.2, 13.4, 13.6, 13.8, 14.0]

# --------------------------------------------------
# Fill panels
# --------------------------------------------------
for i, mvir in enumerate(mass_bins):

    row = i // 3
    col = i % 3

    plot_host_panel_fsub(ax[row, col],trees[i])

    ax[row, col].text(0.02, 0.97, rf"$\log M_{{\rm vir}} = {mvir:.1f}$",
        transform=ax[row, col].transAxes,
        va='top', ha='left')
# --------------------------------------------------
# Axis labels
# --------------------------------------------------
for a in ax[-1, :]:
    a.set_xlabel('cosmic time [Gyr]')

for a in ax[:, 0]:
    a.set_ylabel('f$_{\\rm sub}$')

# --------------------------------------------------
# X limits (adjust as appropriate for your zsample)
# --------------------------------------------------
ax[0,0].set_xlim(0, 10)
ax[0,0].set_yscale("log")
ax[0,0].set_ylim(0, 1)
ax[0,0].legend(loc=3, framealpha=1)
# --------------------------------------------------
# Final layout
# --------------------------------------------------
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    2, 3,
    figsize=(2 * double_textwidth, double_textwidth),
    sharex=True,
    sharey=True
)

mass_bins = [13.0, 13.2, 13.4, 13.6, 13.8, 14.0]

# --------------------------------------------------
# Fill panels
# --------------------------------------------------
for i, mvir in enumerate(mass_bins):

    row = i // 3
    col = i % 3

    filepath = f"../../data/local_trees/evolved_trees/tree_{mvir:.1f}.npz"

    plot_host_panel_MMs(ax[row, col],trees[i])

    ax[row, col].text(0.02, 0.97, rf"$\log M_{{\rm vir}} = {mvir:.1f}$",
        transform=ax[row, col].transAxes,
        va='top', ha='left')
# --------------------------------------------------
# Axis labels
# --------------------------------------------------
for a in ax[-1, :]:
    a.set_xlabel('cosmic time [Gyr]')

for a in ax[:, 0]:
    a.set_ylabel('N$_{\\rm sub}$')

# --------------------------------------------------
# X limits (adjust as appropriate for your zsample)
# --------------------------------------------------
ax[0,0].set_xlim(0, 10)
ax[0,0].set_yscale("log")
ax[0,0].legend(loc=1)
# --------------------------------------------------
# Final layout
# --------------------------------------------------
plt.tight_layout()
plt.show()

In [ ]:
trees[1].canopy(verbose=True)

In [ ]:
dictest.keys()

In [ ]:
trees[4].submass_withering[0]

In [ ]:
trees[4].Nsub_withering[0, :]

In [ ]:
bolshoi = pd.read_csv("../../data/summary_tabs/bolshoi.csv")
satgen = pd.read_csv("../../data/summary_tabs/SatGen_fid.csv")
# ave = pd.read_csv("../../data/summary_tabs/SatGen_ave.csv")
# Mcut = pd.read_csv("../../data/summary_tabs/SatGen_cut.csv")
# Rvir = pd.read_csv("../../data/summary_tabs/SatGen_Rvir.csv")

In [ ]:
b140 = bolshoi[(13.9 < bolshoi["logMvir"]) & (bolshoi["logMvir"] < 14.1)]
b136 = bolshoi[(13.5 < bolshoi["logMvir"]) & (bolshoi["logMvir"] < 13.7)]
b132 = bolshoi[(13.1 < bolshoi["logMvir"]) & (bolshoi["logMvir"] < 13.3)]

In [ ]:
Nsub_132 = 10**b132["logNsub"] / np.average(10**b132["logNsub"])
c_vir_132 = 10**b132["logc"] / np.average(10**b132["logc"])

Nsub_136 = 10**b136["logNsub"] / np.average(10**b136["logNsub"])
c_vir_136 = 10**b136["logc"] / np.average(10**b136["logc"])

Nsub_140 = 10**b140["logNsub"] / np.average(10**b140["logNsub"])
c_vir_140 = 10**b140["logc"] / np.average(10**b140["logc"])

In [ ]:

fig, ax = plt.subplots()

c_smooth = np.linspace(0.3, 3.5)

# KDE plots (store colors explicitly so legend matches)
kde1 = sns.kdeplot(x=c_vir_132, y=Nsub_132, levels=levelz, ax=ax, color="C0")
kde2 = sns.kdeplot(x=c_vir_136, y=Nsub_136, levels=levelz, ax=ax, color="C1")
kde3 = sns.kdeplot(x=c_vir_140, y=Nsub_140, levels=levelz, ax=ax, color="C2")

# Lines
line1, = ax.plot(c_smooth, c_smooth**(-0.86), lw=2, c="k")
line2, = ax.plot(c_smooth, c_smooth**(-0.98), lw=2, c="k", ls=":")

# Custom legend using Line2D
legend_elements = [
    Line2D([0], [0], color="C0", lw=2, label="13.1 < log M$_{\\rm vir}$ < 13.3"),
    Line2D([0], [0], color="C1", lw=2, label="13.5 < log M$_{\\rm vir}$ < 13.7"),
    Line2D([0], [0], color="C2", lw=2, label="13.9 < log M$_{\\rm vir}$ < 14.1"),
    Line2D([0], [0], color="k", lw=2, label="Zentner et al. 2004"),
    Line2D([0], [0], color="k", lw=2, ls=":", label="Alt. slope"),
]

ax.legend(handles=legend_elements)

ax.set_ylabel("N$_{\\rm sub}$ / <N$_{\\rm sub}$>")
ax.set_xlabel("c$_{\\rm vir}$ / <c$_{\\rm vir}$>")
ax.set_xlim(0.2, 2.5)
ax.set_ylim(0.1, 2.5)

ax.set_title("Bolshoi")
plt.show()

In [ ]:
s140 = satgen[(13.9 < satgen["logMvir"]) & (satgen["logMvir"] < 14.1)]
s136 = satgen[(13.5 < satgen["logMvir"]) & (satgen["logMvir"] < 13.7)]
s132 = satgen[(13.1 < satgen["logMvir"]) & (satgen["logMvir"] < 13.3)]

In [ ]:
Nsub_132 = 10**s132["logNsub"] / np.average(10**s132["logNsub"])
c_vir_132 = 10**s132["logc"] / np.average(10**s132["logc"])

Nsub_136 = 10**s136["logNsub"] / np.average(10**s136["logNsub"])
c_vir_136 = 10**s136["logc"] / np.average(10**s136["logc"])

Nsub_140 = 10**s140["logNsub"] / np.average(10**s140["logNsub"])
c_vir_140 = 10**s140["logc"] / np.average(10**s140["logc"])

In [ ]:

fig, ax = plt.subplots()

c_smooth = np.linspace(0.3, 3.5)

# KDE plots (store colors explicitly so legend matches)
kde1 = sns.kdeplot(x=c_vir_132, y=Nsub_132, levels=levelz, ax=ax, color="C0")
kde2 = sns.kdeplot(x=c_vir_136, y=Nsub_136, levels=levelz, ax=ax, color="C1")
kde3 = sns.kdeplot(x=c_vir_140, y=Nsub_140, levels=levelz, ax=ax, color="C2")

# Lines
line1, = ax.plot(c_smooth, c_smooth**(-0.86), lw=2, c="k")
line2, = ax.plot(c_smooth, c_smooth**(-0.98), lw=2, c="k", ls=":")

# Custom legend using Line2D
legend_elements = [
    Line2D([0], [0], color="C0", lw=2, label="log M$_{\\rm vir}$ = 13.2"),
    Line2D([0], [0], color="C1", lw=2, label="log M$_{\\rm vir}$ = 13.6"),
    Line2D([0], [0], color="C2", lw=2, label="log M$_{\\rm vir}$ = 14.0"),
    Line2D([0], [0], color="k", lw=2, label="Zentner et al. 2004"),
    Line2D([0], [0], color="k", lw=2, ls=":", label="Alt. slope"),
]

ax.legend(handles=legend_elements)

ax.set_ylabel("N$_{\\rm sub}$ / <N$_{\\rm sub}$>")
ax.set_xlabel("c$_{\\rm vir}$ / <c$_{\\rm vir}$>")
ax.set_xlim(0.2, 2.5)
ax.set_ylim(0.1, 2.5)

ax.set_title("SatGen")
plt.show()

In [ ]:
def log1pz50_to_ac(log1z50):
    """
    Convert formation redshift z50 to scale factor a_c
    using the Wechsler MAH relation.

    Parameters
    ----------
    z50 : float or array-like
        Redshift where halo reaches half its final mass

    Returns
    -------
    ac : float or array-like
        Characteristic formation scale factor
    """
    z50 = 10**log1z50 - 1
    return np.log(2) / (2 * z50)

ac_132 = log1pz50_to_ac(b132["logz50"].values)
ac_136 = log1pz50_to_ac(b136["logz50"].values)
ac_140 = log1pz50_to_ac(b140["logz50"].values)

ac_132_norm = ac_132 / np.average(ac_132)
ac_136_norm = ac_136 / np.average(ac_136)
ac_140_norm = ac_140 / np.average(ac_140)

Nsub_132 = 10**b132["logNsub"] / np.average(10**b132["logNsub"])
Nsub_136 = 10**b136["logNsub"] / np.average(10**b136["logNsub"])
Nsub_140 = 10**b140["logNsub"] / np.average(10**b140["logNsub"])


In [ ]:
fig, ax = plt.subplots()

c_smooth = np.linspace(0.1, 3.5)


# KDE plots (store colors explicitly so legend matches)
kde1 = sns.kdeplot(x=ac_132_norm, y=Nsub_132, levels=levelz, ax=ax, color="C0")
kde2 = sns.kdeplot(x=ac_136_norm, y=Nsub_136, levels=levelz, ax=ax, color="C1")
kde3 = sns.kdeplot(x=ac_140_norm, y=Nsub_140, levels=levelz, ax=ax, color="C2")


line1, = ax.plot(c_smooth, c_smooth**(0.25), lw=2, c="k")


# Custom legend using Line2D
legend_elements = [
    Line2D([0], [0], color="C0", lw=2, label="13.1 < log M$_{\\rm vir}$ < 13.3"),
    Line2D([0], [0], color="C1", lw=2, label="13.5 < log M$_{\\rm vir}$ < 13.7"),
    Line2D([0], [0], color="C2", lw=2, label="13.9 < log M$_{\\rm vir}$ < 14.1"),
    Line2D([0], [0], color="k", lw=2, label="$\\sim$ Zentner et al. 2004"),
]

# ax.legend(handles=legend_elements)

ax.set_ylabel("N$_{\\rm sub}$ / <N$_{\\rm sub}$>")
ax.set_xlabel("a$_{\\rm c}$ / <a$_{\\rm c}$>")
ax.set_xlim(0.1, 3.5)
ax.set_ylim(0.1, 2.5)
ax.legend(handles=legend_elements, framealpha=1)


ax.set_title("Bolshoi ($a_c = \\frac{\ln 2}{2 z_{50}}$)")
plt.show()

In [ ]:
ac_132 = log1pz50_to_ac(s132["logz50"].values)
ac_136 = log1pz50_to_ac(s136["logz50"].values)
ac_140 = log1pz50_to_ac(s140["logz50"].values)

ac_132_norm = ac_132 / np.average(ac_132)
ac_136_norm = ac_136 / np.average(ac_136)
ac_140_norm = ac_140 / np.average(ac_140)

Nsub_132 = 10**s132["logNsub"] / np.average(10**s132["logNsub"])
Nsub_136 = 10**s136["logNsub"] / np.average(10**s136["logNsub"])
Nsub_140 = 10**s140["logNsub"] / np.average(10**s140["logNsub"])

In [ ]:
fig, ax = plt.subplots()

# KDE plots (store colors explicitly so legend matches)
kde1 = sns.kdeplot(x=ac_132_norm, y=Nsub_132, levels=levelz, ax=ax, color="C0")
kde2 = sns.kdeplot(x=ac_136_norm, y=Nsub_136, levels=levelz, ax=ax, color="C1")
kde3 = sns.kdeplot(x=ac_140_norm, y=Nsub_140, levels=levelz, ax=ax, color="C2")


line1, = ax.plot(c_smooth, c_smooth**(0.25), lw=2, c="k")
c_smooth = np.linspace(0.1, 3.5)


# Custom legend using Line2D
legend_elements = [
    Line2D([0], [0], color="C0", lw=2, label="log M$_{\\rm vir}$ = 13.2"),
    Line2D([0], [0], color="C1", lw=2, label="log M$_{\\rm vir}$ = 13.6"),
    Line2D([0], [0], color="C2", lw=2, label="log M$_{\\rm vir}$ = 14.0"),
    Line2D([0], [0], color="k", lw=2, label="$\\sim$ Zentner et al. 2004"),
]

# ax.legend(handles=legend_elements)

ax.set_ylabel("N$_{\\rm sub}$ / <N$_{\\rm sub}$>")
ax.set_xlabel("a$_{\\rm c}$ / <a$_{\\rm c}$>")
ax.set_xlim(0.1, 3.5)
ax.set_ylim(0.1, 2.5)
ax.legend(handles=legend_elements, framealpha=1)


ax.set_title("SatGen ($a_c = \\frac{\ln 2}{2 z_{50}}$) ")
plt.show()